# K_fit: Kearsley superposition of protein structures

Copyright (C) 2026 Mukundan S. Licensed under the GNU GPL v3.0 or later.
The K_fit engine used by this notebook is MIT licensed.

Fit one or more structures (PDB or mmCIF) onto a reference structure using
exactly the atoms you select. Run the cells from top to bottom.

| Stage | What happens |
|---|---|
| 1 | Install K_fit and its dependencies |
| 2 | Upload files or enter PDB IDs; the first row is the reference; set SOdc / RMSDc |
| 3 | Check the parsed structures |
| 4 | Select the atoms to fit (dropdowns, typed matomids or a pairs CSV) |
| 5 | Run the fitting and view the statistics |
| 6 | Download all outputs as a zip |

Atoms are identified by **matomid** = `model number-atom id`, e.g. `1-245`.

## Stage 1: Install dependencies

In [ ]:
#@title Install K_fit (clone or update the repositories, then pip install)
# Change these URLs if the repositories live elsewhere.
K_FIT_REPO = "https://github.com/bigbigdumdum/K_fit.git"
K_FIT_COLAB_REPO = "https://github.com/bigbigdumdum/K_fit-colab.git"
# Branch installed from both repositories ("main" for the released version).
BRANCH = "test"

import os, subprocess, sys

def clone_or_pull(url, folder, branch):
    """Clone ``branch`` of the repository, or fetch it and switch to its latest version."""
    if os.path.isdir(os.path.join(folder, ".git")):
        subprocess.run(["git", "-C", folder, "fetch", "--depth", "1", "origin", branch], check=True)
        subprocess.run(["git", "-C", folder, "checkout", "-B", branch, "FETCH_HEAD"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", branch, url, folder], check=True)

try:
    import google.colab  # noqa: F401  (only present in a Colab runtime)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Modules imported by an earlier run of this cell stay in memory until the runtime restarts.
ALREADY_IMPORTED = "K_fit" in sys.modules or "k_fit_colab" in sys.modules

def head(folder):
    """Return the commit checked out in ``folder`` (empty if it is not cloned yet)."""
    return subprocess.run(["git", "-C", folder, "rev-parse", "HEAD"],
                          capture_output=True, text=True).stdout.strip()

if IN_COLAB:
    before = [head("/content/K_fit"), head("/content/K_fit-colab")]
    clone_or_pull(K_FIT_REPO, "/content/K_fit", BRANCH)
    clone_or_pull(K_FIT_COLAB_REPO, "/content/K_fit-colab", BRANCH)
    changed = before != [head("/content/K_fit"), head("/content/K_fit-colab")]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "/content/K_fit", "/content/K_fit-colab"], check=True)
    # The version number does not change between commits, so force pip to replace the old copy.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
                    "/content/K_fit", "/content/K_fit-colab"], check=True)
    for folder in ("/content/K_fit", "/content/K_fit-colab"):
        commit = subprocess.run(["git", "-C", folder, "log", "-1", "--format=%h %s"],
                                capture_output=True, text=True).stdout.strip()
        print(f"{os.path.basename(folder)} ({BRANCH}): {commit}")
    if ALREADY_IMPORTED and changed:
        raise SystemExit("K_fit was updated, but this session still runs the old version. "
                         "Use Runtime > Restart session, then run the cells from Stage 2 on.")
else:
    # Local Jupyter: install the two repos yourself, e.g.
    #   pip install -e K_fit -e K_fit-colab
    print("Not running in Colab: skipping clone/install and using the installed packages.")

import K_fit, k_fit_colab, Bio, kearsley_numba
print("K_fit", K_fit.__version__, "| k_fit_colab", k_fit_colab.__version__,
      "| Biopython", Bio.__version__, "| kearsley-numba", kearsley_numba.__version__)

## Stage 2: Input

One row per structure. The **first row is the reference**; every other
structure is fitted onto it. Use **Add row** for more structures. Each row
takes an uploaded file *or* a PDB ID (or the path of a file you uploaded
with the Colab file browser). Duplicate file names are renamed `name_1`,
`name_2`, ...

* **SOdc**: every fitted atom must lie within this distance (A) of its
  reference atom for the fit to be reported (structure overlap = 100 %).
* **RMSDc**: fits with a higher RMSD (A) are reported as NO FIT.

Click **Load structures** when done.

In [ ]:
from k_fit_colab.session import NotebookSession
from k_fit_colab import widgets

session = NotebookSession()
widgets.show(widgets.build_input_panel(session))

## Stage 3: Input checks

In [ ]:
widgets.show_input_summary(session)

## Stage 4: Atom selection

To uniquely identify an atom in a structure file an id called **matomid**, defined as \<model number>-\<atom number>, is used through the notebook

For each structure choose **model / chain**, then a **residue**, then an
**atom** (`name [atom id]`; alternate locations show their occupancy), and
click **Add atom** (**Add N, CA, C** adds the residue's backbone atoms). Residues are
coloured by type, with non-standard residues highlighted in yellow; atoms are
coloured by element (see the legend). Each added atom appears as a row in the
table below (matomid, model, chain, residue, atom). You can also type
matomids and click **Add typed**; **Remove last** and **Clear** edit the
table, and **Copy matomids to** pastes this structure's matomids into another
structure's entry field (check them, then click **Add typed**). Atoms are paired in order: the i-th atom of each target is fitted onto
the i-th atom of the reference.

* **Add another atom set from this structure** copies the structure so a
  second set of atoms from the same file can be fitted onto the first.
* **Fit all models** fits every model of each target separately with the
  same selection. With a single input, every model is fitted onto the model
  you selected.
* **Pairs CSV** skips manual selection. The header row holds the structure
  names (first column = reference), each further row one atom pair of matomids:

```
ref.pdb,model.pdb,1UBQ.cif
1-2,1-5,1-2
1-10,1-13,1-10
```

Click **Check pairs** to print the pairs, errors and warnings.

In [ ]:
widgets.show(widgets.build_selection_panel(session))

## Stage 5: Fitting

In [ ]:
widgets.show(widgets.build_fit_panel(session))

## Stage 6: Download

Downloads `<reference>_vs_<n>structures.zip` (e.g. `1UBQ_vs_3structures.zip`): fitted structures (`*_fit.pdb` / `*_fit.cif`,
coordinates only; header records are removed because they no longer match),
`report.txt`, `transforms.csv` and one `*__pairs.csv` per target.

In [ ]:
widgets.download_results(session)